In [ ]:
import torch #для работы с многомерными массивами
import torch.nn as nn #для построения нейронной сети
import torch.optim as optim #для оптимизации
import random
import numpy as np
try:
    import pygame
except ImportError:
    pygame = None  # Без окна можно выполнять домашку без pygame.
import math

from collections import deque #для памяти


## Гиперпараметры

`N_INPUTS = 9`: пять лучей, скорость, расстояние, sin и cos направления.
`N_ACTIONS = 2`: теперь это **число компонентов действия**, а не количество кнопок.

LR и GAMMA оставляем для будущего обучения; сегодня они не используются для изменения весов. EPS и TARGET_UPDATE убраны из исполняемых настроек: исследование и обновление копий в DDPG будем добавлять отдельно.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

N_INPUTS = 9
N_ACTIONS = 2
HIDDEN = 64

LR = 0.0003
GAMMA = 0.97
BATCH_SIZE = 128
MEMORY_SIZE = 100_000


MAX_SPEED = 10

STUCK_SPEED = 0.05
MAX_STUCK_STEPS = 120

EVAL_EPISODES = 20
EVAL_MAX_STEPS = 3000

RENDER = False
DEMO_MAX_STEPS = 30


Старая сеть выдавала девять Q-оценок, а `argmax` выбирал номер действия. Новый **актор выдаёт сами команды** `(steer, drive)`. Поэтому просто поставить два выхода в старый DQN недостаточно: нужен другой способ обучения.

- Актор: `9 → 64 → 64 → 2`, в конце `Tanh`.
- Критик: наблюдение и действие вместе, `11 → 64 → 64 → 1`, без активации на выходе.

Внутренние слои, `ReLU`, `self.net`, `forward` остаются такими же по устройству, как в нашей QNetwork.
Выход критика — ожидаемая сумма наград с дисконтированием, а не вероятность. До обучения его оценка не имеет надёжного смысла.


## Задание A. Напиши актор

Заполни A1–A4:

- **A1:** число входных признаков первого слоя.
- **A2:** число выходных команд последнего линейного слоя.
- **A3:** активация, ограничивающая эти команды нужным диапазоном.
- **A4:** пропусти `obs` через `self.net` и верни результат.

Подсказки: используй `N_INPUTS`, `N_ACTIONS`, `nn.Tanh()`. В `forward` не нужны `argmax` и перевод в список: здесь результат остаётся тензором, чтобы позднее можно было вычислять градиенты.


In [ ]:
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_INPUTS, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, N_ACTIONS),
            nn.Tanh()  #сжимает от -1 до 1
        )

    def forward(self, obs):
        return self.net(obs)


## Задание C. Напиши критик

Критик принимает **два аргумента**: `obs` размера `[B, 9]` и `action` размера `[B, 2]`.
Чтобы получить вход `[B, 11]`, соедини их **по столбцам**, сохранив строки пачки.

Заполни C1–C3:

- **C1:** сколько признаков у пары «наблюдение + действие»?
- **C2:** соедини `obs` и `action` через `torch.cat(..., dim=...)`.
- **C3:** передай объединённый тензор `x` в `self.net`.

Подсказка: для двумерного тензора `dim=0` — строки, `dim=1` — столбцы. `torch.stack` создаёт новую ось и здесь не нужен. Выход оставляем `[B, 1]`, не используем `squeeze`.


In [ ]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_INPUTS + 2, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, 1),
        )

    def forward(self, obs, action):
        x = torch.cat((obs, action), dim=1)
        return self.net(x)


В нашем `ReplayBuffer` сохранялась пятёрка `(state, action, reward, next_state, done)`. Она остаётся.
Раньше `action` был целым индексом вроде `2`. Теперь это список `[0.2, -0.3]`.

**TODO R1:** замени прежний `torch.LongTensor(np.array(a, dtype=np.int64))` тензором дробных чисел. После выборки его форма должна быть `[batch_size, 2]`.
Подсказка: соседние строки уже показывают, как создать FloatTensor из массива NumPy.

`r` и `d` пока сохраняют прежнюю форму `[B]`. При реализации `learn` согласуем её с выходом критика `[B, 1]`, чтобы не получить ошибочное расширение до `[B, B]`.


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size) #выбираем определённое количество случайных элементов
        s, a, r, s2, d = zip(*batch)
        return (
            torch.FloatTensor(np.array(s)), #тензор из float32
            torch.FloatTensor(np.array(a)),  #R1
            torch.FloatTensor(r),
            torch.FloatTensor(np.array(s2)),
            torch.FloatTensor(d),
        )

    def __len__(self):
        return len(self.buf)


In [ ]:
WIDTH, HEIGHT = 900, 700

LEVEL = 3
if LEVEL == 1:
    walls = [[(50, 100), (850, 100)], [(50, 250), (700, 250)],
             [(850, 100), (850, 600)], [(700, 250), (700, 600)]]
    CHECKPOINTS = [(400, 175), (775, 300)]
    ST_X, ST_Y = 60, 175
elif LEVEL == 2:
    walls = [[(50, 50), (850, 50)], [(850, 50), (850, 650)],
             [(850, 650), (50, 650)], [(50, 650), (50, 50)],
             [(200, 200), (700, 200)], [(700, 200), (700, 500)],
             [(700, 500), (200, 500)], [(200, 500), (200, 200)]]
    CHECKPOINTS = [(450, 575), (775, 350), (450, 125), (125, 350)]
    ST_X, ST_Y = 150, 550
else:
    walls = [[(50, 50), (270, 50)], [(150, 150), (170, 150)],
             [(170, 150), (170, 400)], [(170, 500), (170, 650)],
             [(270, 50), (270, 400)], [(270, 500), (270, 550)],
             [(170, 650), (850, 650)], [(270, 550), (750, 550)],
             [(750, 550), (750, 150)], [(850, 650), (850, 50)],
             [(850, 50), (400, 50)], [(750, 150), (500, 150)],
             [(500, 150), (500, 500)], [(400, 50), (400, 400)],
             [(400, 400), (270, 400)], [(170, 400), (150, 400)],
             [(500, 500), (270, 500)], [(170, 500), (50, 500)],
             [(50, 500), (50, 50)], [(150, 400), (150, 150)]]
    CHECKPOINTS = [(220, 175), (220, 550), (700, 600), (800, 200), (450, 200), (100, 450)]
    ST_X, ST_Y = 100, 100

screen = myfont = clock = None
if RENDER:
    if pygame is None:
        raise ImportError("Установите pygame или поставьте RENDER = False")
    pygame.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Actor-Critic Car: homework")
    myfont = pygame.font.SysFont('Arial', 18)
    clock = pygame.time.Clock()
MAX_RAY_LEN = 200

WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
RED = (250, 0, 0)
GREEN = (0, 250, 0)


Мы уже написали функции на уроке. `target_features` возвращает расстояние, sin и cos относительного направления цели.
Третья функция, `apply_action`, находится внутри нашего класса Car ниже.


In [ ]:
def update_speed(speed, drive):
    speed += drive * 0.1
    return max(0.0, min(speed, MAX_SPEED))


def target_features(x, y, angle, target):
    """Дистанция, sin и cos направления относительно курса автомобиля."""
    cx, cy = target
    dx = cx - x
    dy = y - cy  # На экране y растёт вниз; здесь переводим ось вверх.
    distance = math.hypot(dx, dy)
    if distance < 1e-9:
        # В самой точке направление не определено; выбираем условное значение.
        return [0.0, 0.0, 1.0]
    target_angle = math.atan2(dy, dx)
    relative_angle = target_angle - math.radians(angle)
    return [
        distance / math.hypot(WIDTH, HEIGHT),
        math.sin(relative_angle),
        math.cos(relative_angle),
    ]


`reset`, `drive`, `get_intersection`, `check_collision`, `get_checkpoint_distance`, `get_checkpoint_angle`, `check_checkpoint`, `update_metrics` не изменяем.
В отрисовку добавлены условия `RENDER`, чтобы код работал без окна.

Дополнительно обновляем уже существующий `break_t`, чтобы сохранённый штраф за длительное торможение продолжал работать. Теперь он считает последовательные команды торможения.


In [ ]:
class Car:
    def __init__(self, x, y):
        self.start_x = x
        self.start_y = y
        self.max_speed = MAX_SPEED
        self.rotation_speed = 3
        self.width = 30
        self.height = 15
        self.surface = None
        if RENDER:
            self.surface = pygame.Surface((self.width, self.height), pygame.SRCALPHA)
            self.surface.fill(RED)
        self.reset()

    def reset(self):
        self.x = self.start_x
        self.y = self.start_y
        self.speed = 2.0
        self.angle = 0
        self.alive = True
        self.reward = 0
        self.unreward = 0
        self.break_t = 0
        self.norm_distance = [1.0] * 5
        self.distance = [MAX_RAY_LEN] * 5
        self.current_checkpoint_idx = 0
        # ПРАВКА 8: счётчик "застревания"
        self.stuck_steps = 0
        # ПРАВКА 9: счётчики для честного отчёта метрики на уроке
        self.checkpoints_passed = 0
        self.collisions = 0
        # ПРАВКА 10: для baseline-метрик (сравнение моделей между собой)
        self.total_progress = 0.0
        self.speed_sum = 0.0
        self.steps_alive = 0
        self.max_speed_reached = 0.0

    def drive(self):
        rad = math.radians(self.angle)
        self.x += math.cos(rad) * self.speed
        self.y -= math.sin(rad) * self.speed

    def get_intersection(self, p1, p2, p3, p4):
        x1, y1 = p1; x2, y2 = p2; x3, y3 = p3; x4, y4 = p4
        denom = (y4 - y3) * (x2 - x1) - (x4 - x3) * (y2 - y1)
        if denom == 0: return None
        ua = ((x4 - x3) * (y1 - y3) - (y4 - y3) * (x1 - x3)) / denom
        ub = ((x2 - x1) * (y1 - y3) - (y2 - y1) * (x1 - x3)) / denom
        if 0 <= ua <= 1 and 0 <= ub <= 1:
            return (x1 + ua * (x2 - x1), y1 + ua * (y2 - y1))
        return None

    def cast_rays(self, walls):
        ray_angles = [self.angle - 60, self.angle - 30, self.angle,
                      self.angle + 30, self.angle + 60]
        self.norm_distance = []
        self.distance = []
        for ra in ray_angles:
            rad = math.radians(ra)
            dx = self.x + math.cos(rad) * MAX_RAY_LEN
            dy = self.y - math.sin(rad) * MAX_RAY_LEN
            closest = MAX_RAY_LEN
            for wall in walls:
                pt = self.get_intersection((self.x, self.y), (dx, dy), wall[0], wall[1])
                if pt:
                    d = math.hypot(pt[0] - self.x, pt[1] - self.y)
                    if d < closest: closest = d
            self.norm_distance.append(closest / MAX_RAY_LEN)
            self.distance.append(closest)
            if RENDER:
                color = GREEN if closest > 50 else RED
                pygame.draw.line(screen, color, (self.x, self.y),
                                  (self.x + math.cos(rad) * closest,
                                   self.y - math.sin(rad) * closest))

    def check_collision(self, walls):
        rad = math.radians(self.angle)
        cos_a = math.cos(rad); sin_a = math.sin(rad)
        w = self.width / 2; h = self.height / 2
        corners = [(w, h), (w, -h), (-w, -h), (-w, h)]
        world = [(self.x + lx * cos_a + ly * sin_a,
                  self.y - lx * sin_a + ly * cos_a) for lx, ly in corners]
        sides = [(world[i], world[(i + 1) % 4]) for i in range(4)]
        for side in sides:
            for wall in walls:
                if self.get_intersection(side[0], side[1], wall[0], wall[1]):
                    self.alive = False
                    self.collisions += 1
                    return True
        if not (0 < self.x < WIDTH and 0 < self.y < HEIGHT):
            self.alive = False
            self.collisions += 1
            return True
        return False

    def get_checkpoint_distance(self):
        cx, cy = CHECKPOINTS[self.current_checkpoint_idx]
        return math.hypot(cx - self.x, cy - self.y)

    def get_checkpoint_angle(self):
        cx, cy = CHECKPOINTS[self.current_checkpoint_idx]
        dx = cx - self.x
        dy = self.y - cy
        target_angle = math.degrees(math.atan2(dy, dx))
        diff = target_angle - self.angle
        while diff > 180: diff -= 360
        while diff < -180: diff += 360
        return diff / 180.0

    def check_checkpoint(self):
        if self.get_checkpoint_distance() < 55:
            self.current_checkpoint_idx = (self.current_checkpoint_idx + 1) % len(CHECKPOINTS)
            self.checkpoints_passed += 1
            return 35.0
        return 0.0

    def apply_action(self, action):
        # Функция урока: непрерывный руль и газ/тормоз.
        if not self.alive:
            return
        steer, drive = action
        steer = max(-1.0, min(float(steer), 1.0))
        drive = max(-1.0, min(float(drive), 1.0))
        self.speed = update_speed(self.speed, drive)
        self.angle = (self.angle - steer * self.rotation_speed) % 360.0
        # Сохраняем смысл break_t для нашей прежней функции награды.
        self.break_t = self.break_t + 1 if drive < 0 else 0

    def update_metrics(self):
        self.steps_alive += 1
        self.speed_sum += self.speed
        self.max_speed_reached = max(self.max_speed_reached, self.speed)
        if self.speed < STUCK_SPEED:
            self.stuck_steps += 1
        else:
            self.stuck_steps = 0

    def draw(self):
        if not RENDER:
            return
        rotated = pygame.transform.rotate(self.surface, self.angle)
        screen.blit(rotated, rotated.get_rect(center=(self.x, self.y)))


Сохраняем интерфейс `get_obs`, `choose_action`, `remember`, `save`, `load`.
Вместо `policy_net` и `target_net` теперь есть актор, критик и их две целевые копии. Создание копий через `load_state_dict` тебе уже знакомо.

**Заполни B1–B3 в choose_action:**

- B1: преврати `obs` в FloatTensor и добавь размерность пачки: `[1, 9]`.
- B2: передай `t` в актор.
- B3: из `[1, 2]` получи обычный список из двух чисел: `.squeeze(0).tolist()`.

В этом методе `no_grad()` нужен для выбора действия во время заезда. В `Actor.forward` и `Critic.forward` его ставить нельзя — позже понадобится обучение.

Файлы новых весов имеют другое имя. Старые веса DQN несовместимы с новой архитектурой. Методы save/load сохраняют обе обучаемые сети; это веса для запуска, а не полный снимок процесса обучения.


In [ ]:
class DDPGAgent:
    def __init__(self):
        self.actor = Actor()
        self.critic = Critic()
        self.target_actor = Actor()
        self.target_critic = Critic()
        self.target_actor.load_state_dict(self.actor.state_dict())
        self.target_critic.load_state_dict(self.critic.state_dict())
        self.target_actor.eval()
        self.target_critic.eval()
        for network in (self.target_actor, self.target_critic):
            for parameter in network.parameters():
                parameter.requires_grad_(False)

        self.memory = ReplayBuffer(MEMORY_SIZE)
        self.steps = 0

    def get_obs(self, car):
        target = CHECKPOINTS[car.current_checkpoint_idx]
        return car.norm_distance + [car.speed / MAX_SPEED] + target_features(
            car.x, car.y, car.angle, target
        )

    def choose_action(self, obs):
        with torch.no_grad():
            t = torch.FloatTensor(obs).unsqueeze(0)  #B1
            action = self.actor(t)  #B2
            return action.squeeze(0).tolist()  #B3

    def remember(self, obs, action, reward, next_obs, done):
        self.memory.push(obs, action, reward, next_obs, done)

    def save(self, path="ddpg_weights.pth"):
        torch.save({"actor": self.actor.state_dict(),
                    "critic": self.critic.state_dict()}, path)
        print(f"Модель сохранена {path}")

    def load(self, path="ddpg_weights.pth"):
        weights = torch.load(path, map_location="cpu", weights_only=True)
        self.actor.load_state_dict(weights["actor"])
        self.critic.load_state_dict(weights["critic"])
        self.target_actor.load_state_dict(self.actor.state_dict())
        self.target_critic.load_state_dict(self.critic.state_dict())
        print(f"Модель загружена {path}")

    # learn добавим на следующем занятии. Здесь пока нет обновления весов.


`compute_reward` ниже сохранена без изменения. Не сравнивайте награды этого ноутбука с предыдущей версией домашки: там была упрощённая награда.


In [ ]:
def compute_reward(car, old_dist, new_dist):
    r = 1.1 * car.speed / MAX_SPEED
    r += car.check_checkpoint()

    progress = old_dist - new_dist
    r += max(-2.0, min(2.0, progress * 0.1))

    min_dist = min(car.distance) if car.distance else MAX_RAY_LEN
    if min_dist < 35:
        pen = (35 - min_dist) * 0.15; r -= pen; car.unreward += pen
    if car.break_t > 25:
        r -= 5; car.unreward += 5
    if car.speed == 0:
        r -= 1
    car.reward += r
    return r


## Проверки перед заездом

Сначала заполни A, C, R и B. Здесь проверяются формы, диапазон команд, дробные действия в памяти и сохранённые функции урока.


In [ ]:
assert math.isclose(update_speed(5, 1), 5.1)
assert update_speed(10, 1) == 10
assert update_speed(0.03, -1) == 0
test_car = Car(ST_X, ST_Y)
test_car.speed = MAX_SPEED
test_car.apply_action((1, 1))
assert test_car.speed == MAX_SPEED and test_car.angle == 357
test_car.cast_rays(walls)

agent = DDPGAgent()
obs = agent.get_obs(test_car)
assert len(obs) == N_INPUTS == 9
s = torch.FloatTensor([obs])
with torch.no_grad():
    a = agent.actor(s)
    q = agent.critic(s, a)
assert a.shape == (1, 2) and q.shape == (1, 1)
assert torch.isfinite(a).all() and (a.abs() <= 1).all()
assert torch.isfinite(q).all()
assert isinstance(agent.actor.net[-1], nn.Tanh)
assert isinstance(agent.critic.net[-1], nn.Linear)
for network, sizes in [(agent.actor, [(9, 64), (64, 64), (64, 2)]),
                       (agent.critic, [(11, 64), (64, 64), (64, 1)])]:
    assert [(x.in_features, x.out_features) for x in network.net
            if isinstance(x, nn.Linear)] == sizes

chosen = agent.choose_action(obs)
assert isinstance(chosen, list) and len(chosen) == 2
assert np.allclose(chosen, a.squeeze(0).numpy())

test_memory = ReplayBuffer(10)
test_memory.push(obs, [0.25, -0.75], 1.0, obs, False)
bs, ba, br, bs2, bd = test_memory.sample(1)
assert ba.dtype == torch.float32 and ba.shape == (1, 2)
assert torch.allclose(ba, torch.tensor([[0.25, -0.75]])), "Нельзя обрезать дробное действие до целого"
print("OK: функции урока, сети, выбор действия и память.")


Как раньше, сеть должна сохранять возможность вычислить градиенты. Эта ячейка не обновляет веса — нет optimizer.step(). Подробно разберём на следующем занятии.


In [ ]:
batch_s = torch.randn(4, N_INPUTS)
batch_a = agent.actor(batch_s)
batch_q = agent.critic(batch_s, batch_a)
assert batch_a.shape == (4, 2) and batch_q.shape == (4, 1)
seen = []
hook = agent.critic.net[0].register_forward_pre_hook(
    lambda module, inputs: seen.append(inputs[0].detach().clone())
)
try:
    agent.critic(batch_s, batch_a)
finally:
    hook.remove()
assert torch.allclose(seen[0][:, :N_INPUTS], batch_s)
assert torch.allclose(seen[0][:, N_INPUTS:], batch_a)

(-batch_q.mean()).backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all()
           for p in agent.actor.parameters())
agent.actor.zero_grad(set_to_none=True)
agent.critic.zero_grad(set_to_none=True)
for original, target in [(agent.actor, agent.target_actor), (agent.critic, agent.target_critic)]:
    for p, tp in zip(original.parameters(), target.parameters()):
        assert torch.equal(p, tp) and p.data_ptr() != tp.data_ptr()
print("OK: критик получает действие, градиенты проходят, целевые сети независимы.")


Сравни с прежним циклом: все знакомые вызовы сохранены. Убраны epsilon, agent.learn и автоматическое сохранение; вместо бесконечного while — максимум 30 шагов.

**TODO D1:** получи действие из `agent.choose_action(obs)`.

События окна и рисование тоже сохранены, но выполняются только при RENDER. Если закрыть окно, цикл завершится. Чтобы вновь включить окно, повторно выполни ячейку его создания.

Для памяти `done` в этой заготовке означает терминальное состояние — аварию. Застревание и лимит шагов отмечаем отдельно как `truncated`: это внешняя остановка заезда. При дальнейшем обучении учтём это различие в целевой формуле, не приравнивая все остановки к аварии.


In [ ]:
car = Car(ST_X, ST_Y)
agent = DDPGAgent()
ep_reward = 0.0
history = []
terminated = truncated = False

for step in range(DEMO_MAX_STEPS):
    if RENDER:
        if any(event.type == pygame.QUIT for event in pygame.event.get()):
            print("Окно закрыто, демонстрация остановлена пользователем.")
            break
        screen.fill(WHITE)
        for wall in walls:
            pygame.draw.line(screen, BLACK, wall[0], wall[1])
        for idx, cp in enumerate(CHECKPOINTS):
            is_active = idx == car.current_checkpoint_idx
            pygame.draw.circle(screen, GREEN if is_active else (180, 180, 180), cp, 12)
            if is_active:
                pygame.draw.circle(screen, GREEN, cp, 55, 1)

    car.cast_rays(walls)
    obs = agent.get_obs(car)
    action = agent.choose_action(obs)  #D1
    old_dist = car.get_checkpoint_distance()

    car.apply_action(action)
    car.drive()
    car.cast_rays(walls)
    car.check_collision(walls)

    if car.alive:
        new_dist = car.get_checkpoint_distance()
        car.total_progress += old_dist - new_dist
        step_reward = compute_reward(car, old_dist, new_dist)
        car.update_metrics()
    else:
        step_reward = -350.0

    # После compute_reward: наблюдение уже указывает на новый чекпоинт.
    next_obs = agent.get_obs(car)
    terminated = not car.alive
    truncated = (not terminated) and (
        car.stuck_steps >= MAX_STUCK_STEPS or step + 1 >= DEMO_MAX_STEPS
    )
    ep_reward += step_reward
    agent.remember(obs, action, step_reward, next_obs, terminated)
    history.append((step + 1, *action, car.speed, step_reward))

    if RENDER:
        car.draw()
        screen.blit(myfont.render(f"Step:{step + 1} R:{ep_reward:.1f}", True, BLACK), (5, 5))
        screen.blit(myfont.render(f"steer:{action[0]:+.2f} drive:{action[1]:+.2f}", True, BLACK), (5, 25))
        pygame.display.flip()
        clock.tick(60)
    if terminated or truncated:
        break

print("Шаг, руль, газ/тормоз, скорость, награда:")
for row in history[::5]:
    print(tuple(round(x, 3) for x in row))
print("Переходов в памяти:", len(agent.memory))
print("Чекпоинтов:", car.checkpoints_passed, "Награда:", round(ep_reward, 2))
print("Авария:", terminated, "Внешняя остановка:", truncated)
if RENDER:
    pygame.quit()


Функция сохранена из исходного ноутбука: reward, чекпоинты, прогресс, скорость, столкновения, завершённые круги.
Изменения: choose_action вызывается без eps; лимит можно передать аргументом; запись файла выполняется только при указанном save_path.
Ниже запускаем только 2 заезда по 30 шагов без окна. Это проверка совместимости, не полноценная оценка модели. Низкие метрики сейчас ожидаемы: веса не обучены.


In [ ]:
def evaluate_agent(agent, num_episodes=EVAL_EPISODES, max_steps=EVAL_MAX_STEPS, save_path=None):
    eval_car = Car(ST_X, ST_Y)

    rewards, checkpoints, progresses = [], [], []
    avg_speeds, steps_list, collisions, completed = [], [], [], []

    for _ in range(num_episodes):
        eval_car.reset()
        ep_r = 0.0

        for _ in range(max_steps):
            eval_car.cast_rays(walls)
            obs = agent.get_obs(eval_car)
            action = agent.choose_action(obs)

            old_d = eval_car.get_checkpoint_distance()
            eval_car.apply_action(action)
            eval_car.drive()
            eval_car.cast_rays(walls)
            eval_car.check_collision(walls)

            if not eval_car.alive:
                ep_r += -350.0
                break

            new_d = eval_car.get_checkpoint_distance()
            eval_car.total_progress += (old_d - new_d)
            ep_r += compute_reward(eval_car, old_d, new_d)
            eval_car.update_metrics()

            if eval_car.stuck_steps >= MAX_STUCK_STEPS:
                break

        avg_speed = eval_car.speed_sum / eval_car.steps_alive if eval_car.steps_alive > 0 else 0.0

        rewards.append(ep_r)
        checkpoints.append(eval_car.checkpoints_passed)
        progresses.append(eval_car.total_progress)
        avg_speeds.append(avg_speed)
        steps_list.append(eval_car.steps_alive)
        collisions.append(eval_car.collisions)
        completed.append(eval_car.checkpoints_passed >= len(CHECKPOINTS))

    metrics = {
        "avg_reward": float(np.mean(rewards)),
        "best_reward": float(np.max(rewards)),
        "avg_checkpoints": float(np.mean(checkpoints)),
        "max_checkpoints": int(np.max(checkpoints)),
        "avg_progress": float(np.mean(progresses)),
        "avg_speed": float(np.mean(avg_speeds)),
        "avg_steps": float(np.mean(steps_list)),
        "collision_rate": float(np.mean(collisions)),
        "lap_completion_rate": float(np.mean(completed)),
    }

    print("\n========== BASELINE / EVALUATION ==========")
    print(f"Episodes:              {num_episodes}")
    print(f"Average reward:        {metrics['avg_reward']:.2f}")
    print(f"Best reward:           {metrics['best_reward']:.2f}")
    print(f"Average checkpoints:   {metrics['avg_checkpoints']:.2f}")
    print(f"Max checkpoints:       {metrics['max_checkpoints']}")
    print(f"Average progress:      {metrics['avg_progress']:.2f}")
    print(f"Average speed:         {metrics['avg_speed']:.2f}")
    print(f"Average steps:         {metrics['avg_steps']:.1f}")
    print(f"Collision rate:        {metrics['collision_rate']:.2f}")
    print(f"Lap completion rate:   {100.0 * metrics['lap_completion_rate']:.1f}%")
    print("============================================\n")

    # сохраняем в файл, чтобы потом сравнить несколько моделей между собой
    if save_path is not None:
        np.savez(save_path, **metrics)
    return metrics


In [ ]:
# Прежняя функция cast_rays рисует только при RENDER.
# Для оценки временно отключаем рисование: окно могло уже закрыться.
previous_render = RENDER
RENDER = False
try:
    metrics = evaluate_agent(agent, num_episodes=2, max_steps=30)
finally:
    RENDER = previous_render
assert all(math.isfinite(value) for value in metrics.values())
print("OK: наш evaluate_agent работает с новой архитектурой.")

## Ответь и сохрани ноутбук

1. Почему теперь в ReplayBuffer нужен FloatTensor для действия?
2. Чем отличаются два числа актора от прежних девяти чисел QNetwork?
3. Почему у критика 11 входов и почему на его выходе нет Tanh?
4. Почему next_obs нужно собирать после compute_reward?
5. Какие части нашего старого проекта удалось использовать без переписывания?
6. Что ещё нужно добавить, чтобы сети начали учиться?

**Ответы:**

1. Потому что теперь агент выдаёт действия через дробные числа от -1 до 1.
2. Тем, что все 9 целых чисел QNetwork заменяются 2 float значениями Actor, которые при этом рабоают эффективнее.
3. Потому что он должен оценить как то, что получил актор(9 значений), так и то что он вывел(2 значения). На его выходе нет Tanh потому что он должен выдать оценку, 
не обязательно находящуюся в этом промежутке.
4. Потому что к этому моменту уже изменились показания лучей, скорость и данные о чекпоинте.
5. Почти весь класс car кроме choose_action, compute_reward и большую часть основного цикла, отрисовку трассы.
6. Функцию learn в классе DDPGAgent и функции рассчёта и изменения весов.
